# Instalar os programas e montar o google Drive

In [ ]:
!pip install pyabf
!pip install efel

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

INFORMAR O CAMINHO PARA A PASTA ONDE ESTAO OS ARQUIVOS A SEREM ANALISADOS:


 OBS.: recomendo deixar este arquivo_de_codigo no msm local da pasta dos arquivos

In [ ]:
cd '/content/drive/MyDrive/Laboratorio Neurofisiologia/Projeto Anoxia/'

In [ ]:
ls -li 'P14-P17'

In [ ]:
import os
import efel
import pyabf
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import DataFrame, Series
from scipy.signal import find_peaks
import seaborn as sns
import scipy.stats as stats


In [ ]:
rootFolder = '/content/drive/MyDrive/Laboratorio Neurofisiologia/Projeto Anoxia/'

In [ ]:
rootFolder_controle_CPF = '/content/drive/MyDrive/Laboratorio Neurofisiologia/Projeto Serotonina 2/'

In [ ]:
# Definir nome do protocolo de interesse
protocolo = "Long"

times = []
voltages = []
I = []
cellnames = []
groupnames = []
groupnames2 = []
outliers = ['24722000.abf']

for idade in ['P14-P17']: # ,

  g1 = rootFolder + idade + '/PA- RAMP- PPS - GF'
  if '.' not in g1:
    print(idade)

    for g2 in os.listdir(g1):
      if '.' not in g2:
        print('\t',g2)

        for g3 in os.listdir(g1+'/'+g2):
          if '.' not in g3:
            print('\t\t',g3)


            # -------------------------------------------------------------------------------------------------------------------------- #

            cellnumber = 0
            if 'Cell 01' in os.listdir(g1+'/'+g2+'/'+g3):

              for g4 in os.listdir(g1+'/'+g2+'/'+g3):
                if 'ell' in g4 and '.' not in g4:

                  if 'ANOXIA' in g2 + g3 or 'anoxia' in g2 + g3:
                    if 'CPF' in g2 + g3:
                      grupo = 'CPF_ANOXIA'
                    else:
                      grupo = 'HPC_ANOXIA'

                  elif 'CONTROLE' in g2 + g3 or 'Controle' in g2 + g3:
                    if 'CPF' in g2 + g3:
                      grupo = 'CPF_CONTROLE'
                    else:
                      grupo = 'HPC_CONTROLE'

                  print('\t\t\t',g4, grupo)

                  # -------------------------------------------------------------------------------------------------------------------------- #

                  for g5 in os.listdir(g1+'/'+g2+'/'+g3+'/'+g4):
                    if '.abf' in g5 and g5 not in outliers:

                      firstREO = 0

                      cell = g4

                      filename = g5

                      abf = pyabf.ABF(g1+'/'+g2+'/'+g3+'/'+cell+'/'+filename)

                      fs = abf.dataRate

                      if protocolo in abf.protocol and 'HPC' in grupo and idade+'_'+grupo+'_'+cell not in groupnames2:

                          ############################################
                          # Save Data and plot
                          ############################################

                          print(abf.protocol)
                          print(abf.sweepList)

                          print(grupo+'/'+cell+'/'+filename, abf.protocol, abf.sweepList)
                          # print(abf)

                          plt.figure(figsize=(15, 4))

                          cellnumber = cellnumber + 1
                          celname = 'Cell ' + str(cellnumber)

                          IpA = -130
                          for i in abf.sweepList: # GUI>> modf p dados do gui (muitos steps)

                              abf.setSweep(i, channel=0)
                              IpA = IpA + 20.0

                              peaks, peaksV = find_peaks(abf.sweepY, height=0, distance=40) # V > 0 mV and ISI > 2 ms (at 20kHz)
                              peak_count = len(peaks)

                              if peak_count >= 3 and firstREO == 0:

                                I.append((-110+20*i + abf.sweepC))
                                times.append(abf.sweepX)
                                voltages.append(abf.sweepY)
                                cellnames.append(celname)
                                groupnames.append(idade+'_'+grupo+'_'+cell+'_'+filename)
                                groupnames2.append(idade+'_'+grupo+'_'+cell)

                                firstREO = firstREO + 1

                                plt.plot(abf.sweepX, abf.sweepY, alpha=.95, color = 'blue', linewidth=1.5,
                                          label="I=  %.0f pA" % (IpA))
                                plt.plot(abf.sweepX[peaks], abf.sweepY[peaks], 'o')
                                # b, a = signal.butter(4, 0.05, "low")
                                # current_filtered = signal.filtfilt(b, a, abf.sweepY, padlen=150)
                                # plt.plot(abf.sweepX, current_filtered, alpha=.5, color='r', linewidth=2.5,
                                #           label="I=  %.0f pA" % (IpA))
                                plt.legend()
                                plt.xlim(0,1.100)
                                plt.xlabel('Time (s)')
                                plt.ylabel('Voltage (mV)')
                                # plt.ylim(min(abf.sweepY[5000:15000])-5.0,min(abf.sweepY[5000:15000])+50)
                                plt.title(celname + ": "+(filename))

                          plt.show()
                          plt.close()

In [ ]:
allcells, allcont = np.unique(cellnames,return_counts = True)

In [ ]:
allcells, allcont

In [ ]:
allgroups, allcont = np.unique(groupnames,return_counts = True)

allgroups[0], allcont[0]/1, allgroups[53], allcont[1]/1

In [ ]:
len(allgroups[0:53]), len(allgroups[53:])

In [ ]:
############################################
# Analise eFEL (do Blue Brain Project)
############################################
traces = []

for ii in range(0, len(voltages)):  # MUDEI, era anf.sweepList
    trace = {'T': 1000.0*times[ii], 'V': voltages[ii],
            'I': I[ii],
            'stim_start': [17.2], 'stim_end': [1017.2]}
    traces.append(trace)

Lista_de_medidas = ['time_constant', 'voltage_base', 'steady_state_voltage_stimend', 'time_to_first_spike', 'Spikecount','mean_frequency', 'steady_state_voltage', 'minimum_voltage',
                                        'all_ISI_values', 'peak_time', 'peak_voltage', 'AP_amplitude', 'AP_duration', 'AP_begin_time', 'AP_begin_width', 'AP_duration_half_width',
                                        'min_AHP_values', 'AP_fall_time', 'AP_height', 'AP_rise_time', 'ADP_peak_amplitude', 'ADP_peak_values', 'AHP_depth', 'AHP_depth_abs',
                                        'AHP_depth_abs_slow', 'AHP_depth_from_peak', 'AHP_depth_slow', 'AHP_time_from_peak',
                                        'voltage_deflection', 'decay_time_constant_after_stim', 'ISI_CV', 'adaptation_index']

efel.set_setting("Threshold", 0.0)

feature_values = efel.get_feature_values(traces, Lista_de_medidas, raise_warnings=False);

In [ ]:
ii = 25
I[ii],times[ii],voltages[ii],cellnames[ii],groupnames[ii],groupnames2[ii], feature_values[ii]

In [ ]:
# Set the columns of the output table
table = pd.DataFrame(columns=['cell',
                              'current_pA',
                              'group'])

# Set the columns of the output table
table2 = pd.DataFrame(columns=['cell',
                              'current_pA',
                              'group'])

In [ ]:
group = [grp.split('_')[2] for grp in groupnames2]
idade = [grp.split('_')[0] for grp in groupnames2]
regiao = [grp.split('_')[1] for grp in groupnames2]

In [ ]:
medidas = ['peak_voltage', 'AP_amplitude', 'AP_duration', 'AP_begin_time', 'AP_begin_width', 'AP_duration_half_width',
                                            'min_AHP_values', 'AP_fall_time', 'AP_height', 'AP_rise_time', 'ADP_peak_amplitude', 'ADP_peak_values', 'AHP_depth', 'AHP_depth_abs',
                                            'AHP_depth_abs_slow', 'AHP_depth_from_peak', 'AHP_depth_slow', 'AHP_time_from_peak']
for ii in range(0, len(voltages)):
    length = len(table)

    table.loc[ii, 'cell'] = cellnames[ii]
    table.loc[ii, 'current_pA'] = I[ii][0]
    table.loc[ii, 'group'] =  group[ii]
    table.loc[ii, 'idade'] =  idade[ii]
    table.loc[ii, 'regiao'] =  regiao[ii]

    table2.loc[ii, 'cell'] = cellnames[ii]
    table2.loc[ii, 'current_pA'] = I[ii][0]
    table2.loc[ii, 'group'] =  group[ii]
    table2.loc[ii, 'idade'] =  idade[ii]
    table2.loc[ii, 'regiao'] =  regiao[ii]

    for medida in medidas:

      try:
        table.loc[ii, medida] = feature_values[ii][medida][0:3].mean()
      except:
        print("ERROR")
      table2.loc[ii, medida] = feature_values[ii][medida][0]


In [ ]:
table

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=6, figsize=(24, 12))

data_filtered = table[table['group'].isin(['CONTROLE', 'ANOXIA'])]

for i, medida in enumerate(medidas):


    # Perform t-test
    control_data = data_filtered[data_filtered['group'] == 'CONTROLE'][medida].astype(float) # Convert to numeric type
    anoxia_data = data_filtered[data_filtered['group'] == 'ANOXIA'][medida].astype(float) # Convert to numeric type
    t_stat, p_value = stats.ttest_ind(control_data, anoxia_data)

    # print(f"T-statistic: {t_stat:.2f}")
    # print(f"P-value: {p_value:.3f}")

    # plt.title(f"Mean {medida}, P-value: {p_value:.3f}")
    # Create boxplot
    row = i // 6  # Calculate the row index
    col = i % 6   # Calculate the column index
    ax = axes[row, col]  # Access the correct Axes object using row and column indices
    ax.set_title(f"P-value: {p_value:.3f}")
    sns.boxplot(x='group', y=medida, data=data_filtered, ax = ax)

plt.tight_layout()
plt.show()

In [ ]:
Control = table.get(table["group"] == "CONTROLE")
DHB = table.get(table["group"] == "ANOXIA")
Control

In [ ]:
ls 'Analises_Fernando'

In [ ]:
DHB.to_csv('Analises_Fernando/Anx_p15_ativas_3spikes.csv')
Control.to_csv('Analises_Fernando/Ctl_p15_ativas_3spikes.csv')

In [ ]:
sns.histplot(y='current_pA', data=Control, bins=range(6,300,5))
sns.histplot(y='current_pA', data=DHB, bins=range(10,300,5))

## First Spike

In [ ]:
table2

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=6, figsize=(24, 12))

data_filtered = table2[table['group'].isin(['CONTROLE', 'ANOXIA'])]

for i, medida in enumerate(medidas):


    # Perform t-test
    control_data = data_filtered[data_filtered['group'] == 'CONTROLE'][medida].astype(float) # Convert to numeric type
    anoxia_data = data_filtered[data_filtered['group'] == 'ANOXIA'][medida].astype(float) # Convert to numeric type
    t_stat, p_value = stats.ttest_ind(control_data, anoxia_data)

    # print(f"T-statistic: {t_stat:.2f}")
    # print(f"P-value: {p_value:.3f}")

    # plt.title(f"Mean {medida}, P-value: {p_value:.3f}")
    # Create boxplot
    row = i // 6  # Calculate the row index
    col = i % 6   # Calculate the column index
    ax = axes[row, col]  # Access the correct Axes object using row and column indices
    ax.set_title(f"P-value: {p_value:.3f}")
    sns.boxplot(x='group', y=medida, data=data_filtered, ax = ax)

plt.tight_layout()
plt.show()

In [ ]:
Control = table.get(table2["group"] == "CONTROLE")
DHB = table.get(table2["group"] == "ANOXIA")
Control

In [ ]:
ls 'Analises_Fernando'

In [ ]:
DHB.to_csv('Analises_Fernando/Anx_p15_ativas_1spike.csv')
Control.to_csv('Analises_Fernando/Ctl_p15_ativas_1spike.csv')

In [ ]:
sns.histplot(y='current_pA', data=Control, bins=range(6,300,5))
sns.histplot(y='current_pA', data=DHB, bins=range(10,300,5))